In [4]:
from mypy import biostat as bst
import pandas as pd
import statsmodels as stats
import numpy as np

def fc_pv_calc(subdf):
    from scipy import stats
    mean_ctrl = subdf[subdf['TB Classification']
                      == 'Unlikely TB']['value'].mean()
    fc, pv = [], []
    mean_tr1 = subdf[subdf['TB Classification']
                     == 'Confirmed TB']['value'].mean()
    fc = mean_tr1 - mean_ctrl
    pv = stats.ttest_ind(subdf[subdf['TB Classification'] == 'Confirmed TB']['value'],
                         subdf[subdf['TB Classification']
                               == 'Unlikely TB']['value']
                         )[1]
    return pd.DataFrame([fc, pv]).T

dd = pd.read_csv('processed/protein_level_melted_merged.csv')
dd = dd[['Protein.Group', 'variable', 'value', 'COMBO Plasma Box Number',
         'Age', 'TB Classification', 'HIV Status', 'Genes']]


stats_df = dd.groupby(
    ['Protein.Group', 'Genes']).apply(fc_pv_calc).reset_index()
stats_df.columns = ['PID', 'GN', 'idx', 'Log2FC', 'p']
stats_df['q'] = bst.multiple_testing_correction(stats_df['p'].values)
stats_df.drop(['idx'], axis=1, inplace=True)
#stats_df['Genes'] = stats_df['PID'].map(mp)
stats_df['logq'] = -np.log10(stats_df['q'])
stats_df['issign'] = (stats_df['q'] < 0.05) & (
    np.abs(stats_df['Log2FC']) > 0.5)
stats_df.to_csv('figures/fig2/data/Confirmed_vs_Unlikely.csv', index=False)


In [5]:
stats_df = stats_df[stats_df['q']<=0.05]
stats_df['uu'] = ['up' if x>0 else 'down' for x in stats_df['Log2FC']]
stats_df.groupby(['uu']).size()
for x in 'up', 'down':
    stats_df[stats_df['uu']==x].to_csv('figures/fig2/data/{}.csv'.format(x))

In [17]:
## pie chart
import matplotlib.pyplot as plt
from matplotlib import rc
import pandas as pd

def autopct_format(values):
    def my_format(pct):
        total = sum(values)
        val = int(round(pct*total/100.0))
        return '{v:d}'.format(v=val)
    return my_format



dd = pd.read_csv('figures/fig3/data/idmapping_2023_07_04.csv')
dd['sec'] = [x.split(' ')[2] for x in dd['Subcellular location [CC]']]
dd['sec']=dd['sec'].str.replace('[', '')
dd['sec']=dd['sec'].str.replace(r'.', '')
dd['sec']=dd['sec'].str.replace(r',', '')
dd['sec']=dd['sec'].str.replace(r';', '')
dd['sec'].replace({'Cell': 'Membrane', 'Isoform':'Cytoplasm', 'Soluble':'Secreted', 'Endoplasmic':'ER'}, inplace=True)
dd = dd.groupby('sec').size().to_frame().reset_index()


# rc('font', **{'family': 'sans-serif', 'sans-serif': ['Helvetica']})
# rc('text', usetex=True)

# fig, ax = plt.subplots(figsize=(1.5, 1.5))

# patches, texts, autotexts = ax.pie(
#     dd[0], labels=dd['sec'], autopct=autopct_format(list(dd[0])))
# fig.savefig("figures/fig3/secr_piechart.pdf", dpi=600, bbox_inches='tight')
# fig.savefig("figures/fig3/secr_piechart.svg", format="svg")
# plt.close()

# dd = pd.read_csv('figures/fig3/data/idmapping_2023_07_04.csv')
# dd['sec'] = [x.split(' ')[2] for x in dd['Subcellular location [CC]']]
# dd.to_csv('test_sect.csv')
dd2 = pd.read_csv('figures/fig3/data/Confirmed_vs_Unlikely.csv')
dd2= dd2[dd2['q']<0.05]

dd3 = pd.read_csv('meta/subcellular_location/subcellular_location_data.tsv', sep='\t')
dd3 =dd3[['Gene name', 'Main location', 'Additional location', 'Extracellular location']]
dd3['Gene name']=dd3['Gene name'].str.upper()
dd2 = pd.merge(dd3,dd2, right_on='GN', left_on='Gene name', how='right')
dd2.to_csv('tmp/testlocalization.csv')

/var/folders/6w/j26fwf3s1gxbgknbxwsd30xh0000gn/T/ipykernel_43232/2551570176.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dd['sec'].replace({'Cell': 'Membrane', 'Isoform':'Cytoplasm', 'Soluble':'Secreted', 'Endoplasmic':'ER'}, inplace=True)
